In [2]:
'''
Univariate-related functions:
[1] OptimizeDtypes
[2] CompareDistributions


Authors: Danusorn Sitdhirasdr <danusorn.si@gmail.com>
versionadded:: 10-05-2026

'''
import numpy as np, pandas as pd, time
import scipy.stats as stats
from warnings import warn
from collections import namedtuple, OrderedDict
from dataclasses import dataclass

import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.transforms as transforms

from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.utils import column_or_1d, check_consistent_length
from sklearn.utils.validation import check_array
from sklearn.preprocessing import QuantileTransformer

from scipy.stats import wasserstein_distance
from scipy.spatial.distance import jensenshannon
from scipy.stats.distributions import kstwobign

__all__ = ["OptimizeDtypes", 
           "CompareDistributions"]

In [3]:
class ValidateParams:
    
    '''Validate parameters'''
    
    def Interval(self, Param, Value, dtype=int, 
                 left=None, right=None, closed="both"):

        '''
        Validate numerical input.

        Parameters
        ----------
        Param : str
            Parameter's name

        Value : float or int
            Parameter's value

        dtype : {int, float}, default=int
            The type of input.

        left : float or int or None, default=None
            The left bound of the interval. None means left bound is -∞.

        right : float, int or None, default=None
            The right bound of the interval. None means right bound is +∞.

        closed : {"left", "right", "both", "neither"}
            Whether the interval is open or closed. Possible choices are:
            - "left": the interval is closed on the left and open on the 
              right. It is equivalent to the interval [ left, right ).
            - "right": the interval is closed on the right and open on the 
              left. It is equivalent to the interval ( left, right ].
            - "both": the interval is closed.
              It is equivalent to the interval [ left, right ].
            - "neither": the interval is open.
              It is equivalent to the interval ( left, right ).

        Returns
        -------
        Value : float or int
            Parameter's value

        '''
        Options = {"left"    : (np.greater_equal, np.less), # a<=x<b
                   "right"   : (np.greater, np.less_equal), # a<x<=b
                   "both"    : (np.greater_equal, np.less_equal), # a<=x<=b
                   "neither" : (np.greater, np.less)} # a<x<b

        f0, f1 = Options[closed]
        c0 = "[" if f0.__name__.find("eq")>-1 else "(" 
        c1 = "]" if f1.__name__.find("eq")>-1 else ")"
        v0 = "-∞" if left is None else str(dtype(left))
        v1 = "+∞" if right is None else str(dtype(right))
        if left  is None: left  = -np.inf
        if right is None: right = +np.inf
        interval = ", ".join([c0+v0, v1+c1])
        tuples = (Param, dtype.__name__, interval, Value)
        err_msg = "%s must be %s or in %s, got %s " % tuples    

        if isinstance(Value, dtype):
            if not (f0(Value, left) & f1(Value, right)):
                raise ValueError(err_msg)
        else: raise ValueError(err_msg)
        return Value

    def StrOptions(self, Param, Value, options, dtype=str):

        '''
        Validate string or boolean inputs.

        Parameters
        ----------
        Param : str
            Parameter's name
            
        Value : float or int
            Parameter's value

        options : set of str
            The set of valid strings.

        dtype : {str, bool}, default=str
            The type of input.
        
        Returns
        -------
        Value : float or int
            Parameter's value

        '''
        if Value not in options:
            err_msg = f'{Param} ({dtype.__name__}) must be either '
            for n,s in enumerate(options):
                if n<len(options)-1: err_msg += f'"{s}", '
                else: err_msg += f' or "{s}" , got %s'
            raise ValueError(err_msg % Value)
        return Value
    
    def check_range(self, param0, param1):
        
        '''
        Validate number range.
        
        Parameters
        ----------
        param0 : tuple(str, float)
            A lower bound parameter e.g. ("name", -100.)
            
        param1 : tuple(str, float)
            An upper bound parameter e.g. ("name", 100.)
            
        '''
        if param0[1] >= param1[1]:
            raise ValueError(f"`{param0[0]}` ({param0[1]}) must be less"
                             f" than `{param1[0]}` ({param1[1]}).")
            
    def check_y_inputs(self, y_proba, y_true=None):
        
        '''
        Validate inputs for calibration models.

        Parameters
        ----------
        y_proba : array-like, shape (n_samples,)
            Predicted probabilities ranges from 0 to 1.
            
        y_true : array-like, shape (n_samples,), default=None
            True binary labels (0/1).
            
        Returns
        -------
        y_proba : ndarray shape (n_samples,)
            Validated probabilities.
            
        y_true : ndarray shape (n_samples,)
            Validated labels. This returns None when `y_ture` is not 
            provided.
            
        '''
        # Ensure shapes
        y_proba = column_or_1d(y_proba)
        y_proba = check_array(y_proba, ensure_2d=False, dtype=float)

        # Probability range
        eps = np.finfo(float).eps
        y_proba = np.clip(y_proba, eps, 1 - eps)
        
        if y_true is not None:
            
            # Check lengths
            y_true = column_or_1d(y_true)
            check_consistent_length(y_proba, y_true)

            # Check that y_true only contains {0,1}
            unique_labels = np.unique(y_true)
            if not np.all(np.isin(unique_labels, [0, 1])):
                raise ValueError("`y_true` must be binary (0 or 1).")
            return y_proba, y_true
        
        else: return y_proba, None
        
    def check_class_weight(self, class_weight):
        
        '''
        Validate the `class_weight` parameter.

        Parameters
        ----------
        class_weight : {dict, "balanced"} or None
            Weights associated with classes in the form `{label: weight}`.
            If set to "balanced", the weights are automatically adjusted
            inversely proportional to class frequencies.

        Returns
        -------
        class_weight : dict, str or None
            Normalized class_weight.
            
        '''
         # Validate the `class_weight` parameter.
        if class_weight is None:
            return None
        elif isinstance(class_weight, str):
            args = ("class_weight", class_weight, ["balanced"], str)
            return self.StrOptions(*args)
        elif isinstance(class_weight, dict):
            return class_weight
        else:raise ValueError(f"Invalid value for `class_weight`: {class_weight}. "
                              f"Expected dict, 'balanced', or None.")
            
    def check_estimator(self, estimator):
        
        '''
        Validate that an estimator follows sklearn interface.

        Parameters
        ----------
        estimator : object
            Estimator instance.

        Raises
        ------
        ValueError
            If estimator does not implement required methods/attributes.
            
        '''
        for m in ["get_params", "set_params", "fit", "predict", "predict_proba"]:
            if not callable(getattr(estimator, m, None)):
                raise ValueError(f"Estimator missing required method `{m}()`.")

        if estimator.get_params().get("fit_intercept", None) is None:
            raise ValueError("Estimator missing required parameter "
                             "`fit_intercept` in get_params().")
        
        # Fit model with dummy X and y
        N, rnd = 10, np.random.RandomState(0)
        estimator.fit(rnd.rand(N).reshape(-1,1), 
                      rnd.randint(0, 2, size=N))
        
        for a in ["coef_", "intercept_"]:
            if getattr(estimator, a, None) is None:
                raise ValueError(f"Estimator missing required attribute "
                                 f"`{attr}` after fit().")

    def check_bins(self, bins):
        
        '''
        Validate 'bins'

        Parameters
        ----------
        bins : int, sequence
            Number of bins or sequence to divide probabilities.
            
        Returns
        -------
        bins : int, sequence
            
        '''
        # Case 1: bins is integer
        if isinstance(bins, int):
            if bins <= 0:
                raise ValueError(f"Number of bins must be positive,"
                                 f" got {bins}.")

        # Case 2: bins is a sequence of bin edges
        elif isinstance(bins, (list, np.ndarray)):
            
            bins = np.asarray(bins)
            if bins.ndim != 1:
                raise ValueError("Bin edges must be a 1D sequence")
            elif not np.all(np.isfinite(bins)):
                raise ValueError("Bin edges contain non-finite values (NaN/inf)")
            elif not np.all(np.diff(bins) > 0):
                raise ValueError("Bin edges must be strictly increasing")
            elif len(bins) < 2:
                raise ValueError("At least two bin edges are required")
            else: pass

        else: raise TypeError("bins must be int or sequence")
        
        return bins

In [4]:
class OptimizeDtypes(BaseEstimator, TransformerMixin, ValidateParams):
    
    '''
    Optimize DataFrame column dtypes for improved memory efficiency
    while preserving data integrity and usability.

    Parameters
    ----------
    max_category : int, default=100
        Maximum number of unique non-null values allowed for converting
        object columns to categorical dtype.

    max_category_ratio : float or None, default=None
        Maximum allowed ratio of unique non-null values relative to the
        total number of non-null observations for categorical conversion.

        Example:
        - 0.05 means convert to category only if unique_ratio <= 5%

        If None, ratio-based filtering is disabled.

    float_tolerance : float, default=1e-8
        Absolute tolerance used to determine whether floating-point
        values are effectively integer-like during dtype inference.

    Notes
    -----
    - NaN values are preserved.
    - Float downcasting considers numeric range only, not precision loss.
    - Integer-like float columns (e.g. [1.0, 2.0]) may be converted to
      integer dtypes when possible.
    - Datetime and timedelta columns are preserved unchanged.
    
    '''
    
    # Numerical types
    num_types = {"iinfo" : [np.int8, np.int16, np.int32, np.int64],
                 "finfo" : [np.float16, np.float32, np.float64]}

    def __init__(self, max_category=100, max_category_ratio=None, 
                 float_tolerance=1e-8):
        
        # Validate all parameters
        self.max_category = self.Interval("max_category", 
                                          max_category, 
                                          int, 2, None, "left")
      
        if max_category_ratio is not None:
            self.max_category_ratio = self.Interval("max_category_ratio", 
                                                    max_category_ratio, 
                                                    float, 0., 1., "both")
        else: self.max_category_ratio = max_category_ratio
        
        eps = np.finfo(float).eps
        self.float_tolerance = self.Interval("float_tolerance", 
                                             float_tolerance, float, 
                                             eps, 1 - eps, "both")
                
    def fit(self, X):

        '''
        This function analyzes each column in the input DataFrame and
        determines the most memory-efficient dtype that preserves the
        underlying data characteristics. The inferred dtypes are stored
        in `self.dtypes_` and later applied using `transform()`.

        Conditions:
        - Integer-like numeric columns → smallest compatible integer dtype
        - Continuous numeric columns → smallest compatible float dtype
        - Boolean columns → int8
        - Low-cardinality object columns → category
        - High-cardinality object columns → object
        - Datetime and timedelta columns → preserved unchanged

        Parameters
        ----------
        X : pd.DataFrame
            Input DataFrame used to infer optimized column dtypes.

        Returns
        -------
        self
            Fitted dtype optimizer.

        Attributes
        ----------
        dtypes_ : dict
            Dictionary mapping column names to inferred optimized dtypes.

        Notes
        -----
        - This method only learns optimized dtypes and does not modify
          the input DataFrame.
        - Columns containing only missing values are ignored.
        - Integer-like detection uses `float_tolerance` to account for
          floating-point precision noise.
        - Object columns may be converted to categorical dtype depending
          on `max_category` and `max_category_ratio`.

        '''

        if not isinstance(X, pd.DataFrame):
            raise TypeError(f"`X` must be pd.DataFrame. Got {type(X)}.")
        
        self.dtypes_ = dict()
        
        for col in X.columns:
            
            x = X[col]

            # Preserve datetime / timedelta
            if (pd.api.types.is_datetime64_any_dtype(x) or 
                pd.api.types.is_timedelta64_dtype(x)):
                
                continue

            # Boolean
            elif pd.api.types.is_bool_dtype(x):
                
                self.dtypes_[col] = np.int8

            # Numeric
            elif pd.api.types.is_numeric_dtype(x):

                # Skip fully missing columns
                a = x.to_numpy(dtype=np.float64)
                if np.isnan(a).all(): continue

                # Determine integer-like
                kwargs = dict(atol=self.float_tolerance, equal_nan=True)
                is_int = np.all(np.isclose(a, np.round(a), **kwargs))
                info_type = "iinfo" if is_int else "finfo"
                
                # Max-Min range
                a_min = np.nanmin(a)
                a_max = np.nanmax(a)

                # Numerical optimization
                for dtype in self.num_types[info_type]:
                    limits = getattr(np, info_type)(dtype)
                    if (a_min >= limits.min) & (a_max <= limits.max): 
                        attr = f"{dtype.__name__.capitalize()}Dtype" 
                        self.dtypes_[col] = getattr(pd, attr)()
                        break

            # Object / string 
            elif pd.api.types.is_object_dtype(x):

                # Check number of unique elements in x
                n_unique = x.nunique(dropna=True)
                n_total = x.notna().sum()
                use_category = n_unique <= self.max_category
                
                # Determine ratio of uniqueness
                if self.max_category_ratio is not None:
                    unique_ratio = n_unique / n_total if n_total > 0 else 1
                    use_category &= (unique_ratio <= self.max_category_ratio)
                
                if use_category: self.dtypes_[col] = "category"
                else: self.dtypes_[col] = "object"

            # Category
            elif isinstance(x.dtype, pd.CategoricalDtype):
                
                self.dtypes_[col] = "category"
        
        return self
    
    def transform(self, X):

        '''
        Transform DataFrame columns to the optimized dtypes determined
        during fitting.

        Parameters
        ----------
        X : pd.DataFrame
            Input DataFrame to transform.

        Returns
        -------
        X_transformed : pd.DataFrame
            DataFrame with converted column dtypes based on `self.dtypes_`.
        
        Attributes
        ----------
        memory_report : dict
            Summary of DataFrame memory usage before and after dtype
            optimization. Memory statistics are reported in megabytes,
            where 1 MB = 1024² bytes.

            Contains:
            - before_mb : Memory usage of the original DataFrame.
            - after_mb  : Memory usage of the optimized DataFrame.
            - delta_mb  : Reduction in memory usage.
            - delta_pct : Percentage reduction in memory usage.
            
        failed_dtypes_ : dict
            Dictionary containing columns whose dtype optimization failed
            during transformation, along with the corresponding error
            messages and original dtypes.
            
        Notes
        -----
        - Only columns stored in `self.dtypes_` are transformed.
        - Columns not seen during fitting are preserved unchanged.
        - Conversion failures are ignored and original dtype is retained.
        
        '''
        # Validation
        if not hasattr(self, "dtypes_"):
            raise AttributeError("This transformer is not fitted yet. "
                                 "Call `fit()` before `transform()`.")

        if not isinstance(X, pd.DataFrame):
            raise TypeError(f"`X` must be pd.DataFrame. Got {type(X)}.")

        X_transformed = X.copy()
  
        # Apply learned dtypes
        self.failed_dtypes_ = dict()
        for col, dtype in self.dtypes_.items():
            
            if col not in X_transformed.columns:
                continue
            
            try: X_transformed[col] = X_transformed[col].astype(dtype)
            except Exception as e:
                warn(f"Column '{col}' could not be converted "
                     f"to {dtype}. Original dtype "
                     f"'{X_transformed[col].dtype}' was retained. "
                     f"Reason: {e}", UserWarning)
                
                self.failed_dtypes_[col] = {
                    "target_dtype"  : str(dtype),
                    "original_dtype": str(X_transformed[col].dtype),
                    "reason"        : str(e)}
        
        # DataFrame memory usage
        self.memory_report = self.memory_usage(X, X_transformed)
    
        return X_transformed

    def fit_transform(self, X):

        '''
        Fit the transformer and apply dtype optimization to the input
        DataFrame.

        Parameters
        ----------
        X : pd.DataFrame
            Input DataFrame.

        Returns
        -------
        X_transformed : pd.DataFrame
            DataFrame with optimized column dtypes.
            
        '''
        return self.fit(X).transform(X)

    def memory_usage(self, df_before, df_after):
        
        '''
        Generate memory usage comparison report between two DataFrames.

        Parameters
        ----------
        df_before : pd.DataFrame
            Original DataFrame before optimization.

        df_after : pd.DataFrame
            Optimized DataFrame after dtype conversion.

        Returns
        -------
        memory_report : dict
            Summary of DataFrame memory usage before and after dtype
            optimization. Memory statistics are reported in megabytes,
            where 1 MB = 1024² bytes.

            Contains:
            - before_mb : Memory usage of the original DataFrame.
            - after_mb  : Memory usage of the optimized DataFrame.
            - delta_mb  : Reduction in memory usage.
            - delta_pct : Percentage reduction in memory usage.
            
        '''
        # Convert bytes → MB        
        before_mb = df_before.memory_usage(deep=True).sum() / 1024**2
        after_mb  = df_after.memory_usage(deep=True).sum() / 1024**2
        
        # Optimized delta
        delta_mb  = before_mb - after_mb
        delta_pct = round(delta_mb / before_mb,4) if before_mb > 0 else 0
        num_format = lambda x : float(np.round(x, 4))
        
        return {"before_mb": num_format(before_mb),
                "after_mb" : num_format(after_mb),
                "delta_mb" : num_format(delta_mb),
                "delta_pct": num_format(delta_pct)*100}

In [5]:
class Binning:
    
    @staticmethod
    def create_numeric_bins(x, n_bins=10, equal_width=True, normal_dist=False):
        
        '''
        Generate bin edges using equal-width or quantile-based binning.

        This function creates a monotonically increasing set of bin edges 
        from input data `x`. Optionally, the data can be transformed to a 
        normal distribution prior to binning to improve spacing in skewed 
        distributions.

        Two binning strategies are supported:
        - Equal-width: divides the range into uniform intervals
        - Quantile-based: ensures approximately equal sample sizes per bin

        The final bin edge is adjusted with a small epsilon to ensure the 
        maximum value is always captured.

        Parameters
        ----------
        x : np.ndarray
            Input data array (1D). May contain NaN values.

        n_bins : int, default=10
            Number of desired bins (must be >= 2).

        equal_width : bool, default=True
            If True, uses equal-width binning. Otherwise, uses quantile-based 
            binning.

        normal_dist : bool, default=True
            If True, applies a quantile transformation to approximate a normal 
            distribution before binning.

        Returns
        -------
        bin_edges : np.ndarray
            Monotonically increasing array of bin edges with length (n_bins + 1).

        Notes
        -----
        - NaN values are ignored when computing bin edges.
        - Quantile binning may produce fewer bins if duplicate values exist.
        - The transformation (if applied) is reversed before returning bin 
          edges.
        
        '''
        # Pre-validation
        if not isinstance(n_bins, int) or n_bins <= 2:
            raise ValueError(f"`n_bins` must be an integer > 2. Got {n_bins}.")
        
        # Accept only non-NaN values
        x = np.asarray(x).ravel()
        x_valid = x[~np.isnan(x)]

        if x_valid.size < 2:
            raise ValueError("`x` must contain more than 1 non-NaN value.")

        # Optional transformation
        if normal_dist:
            qt = QuantileTransformer(n_quantiles=min(10000, x_valid.size),
                                     output_distribution="normal",
                                     random_state=0)
            x_trans = qt.fit_transform(x_valid.reshape(-1, 1)).ravel()
        else: x_trans = x_valid

        # Binning
        if equal_width:
            args = (np.min(x_trans), np.max(x_trans), n_bins + 1)
            bin_edges = np.linspace(*args)
        else:
            quantiles = np.linspace(0, 100, n_bins + 1)
            bin_edges = np.unique(np.percentile(x_trans, quantiles))

        # Inverse transform
        if normal_dist:
            bin_edges = bin_edges.reshape(-1, 1)
            bin_edges = qt.inverse_transform(bin_edges).ravel()

        # Ensure last bin captures max
        bin_edges[-1] += np.finfo("float64").eps

        return bin_edges
    
    @staticmethod
    def compute_frequency(x, bin_edges, include_nan=True, n_min=30, merge=True):
        
        '''
        Compute frequency distribution with adaptive bin merging.

        This function assigns each non-NaN value in `x` to bins defined by 
        `bin_edges` and computes the frequency per bin. Values outside the 
        range are capped to the nearest valid bin.

        To ensure statistical reliability, bins with low sample counts are 
        iteratively merged based on a minimum threshold (`n_min`):
        - The bin with the smallest frequency is selected.
        - It is merged with an adjacent bin by removing a bin edge.
        - The merge direction favors the smaller neighboring bin to preserve 
          local structure and maintain resolution.

        The process continues until:
        - All bins meet the minimum sample requirement, or
        - Only two bins remain (minimum allowed)

        Parameters
        ----------
        x : np.ndarray
            Input data array (1D). May contain NaN values.

        bin_edges : np.ndarray
            Monotonically increasing array of bin edges with length (n_bins + 1).

        include_nan : bool, default=True
            If True, appends the count of NaN values as the last element of 
            the frequency array.
        
        n_min : int or float, default=30
            Minimum required observations per bin when `merge=True`.
            - If int >= 1: Minimum absolute count per bin.
            - If 0 < float < 1: Minimum fraction of total non-NaN observations.

        merge : bool, default=True
            If True, performs iterative adaptive bin merging based on `n_min`. 
            Otherwise, frequencies are computed directly from the provided bin 
            edges without modification.

        Returns
        -------
        freq : np.ndarray
            Frequency of observations per bin after merging. If `include_nan=True`,
            the last element represents NaN count.

        bin_edges : np.ndarray
            Final bin edges after optional adaptive merging.

        Notes
        -----
        - Bin intervals follow [edge_i, edge_{i+1}) convention.
        - The number of bins is len(bin_edges) - 1.
        - Empty bins are preserved with zero counts.
        - The algorithm guarantees at least two bins in the output.
        - Merging prioritizes structural preservation over aggressive smoothing.

        '''
        
        def get_counts(x, edges):
            n_bins = len(edges) - 1
            # Digitize (right=False : bins[i-1] <= x < bins[i])
            idx = np.digitize(x, edges, right=False)
            idx = np.clip(idx, 1, n_bins)
            return np.bincount(idx - 1, minlength=n_bins)

        # Accept only non-NaN values
        x = np.asarray(x).ravel()
        x_valid = x[~np.isnan(x)]
        
        n_total = x_valid.size
        if n_total == 0: return None, bin_edges 
        
        # Interpret n_min
        if isinstance(n_min, float):
            if not (0 < n_min < 1):
                raise ValueError("If float, `n_min` must be between 0 and 1.")
            min_count = int(np.ceil(n_min * n_total))
        elif isinstance(n_min, int):
            if n_min < 1: raise ValueError("If int, `n_min` must be >= 1.")
            min_count = n_min
        else: raise TypeError("`n_min` must be int or float.")

        # Iterative merging
        if merge:
            
            while True:

                # Calculate count in respective bins
                counts = get_counts(x_valid, bin_edges)

                # Stop if already only 2 bins
                if len(bin_edges) <= 3: break

                # Find smallest bin
                idx = np.argmin(counts)

                # Stop if all bins satisfy threshold
                if counts[idx] >= min_count: break

                # Decide merge direction
                if idx == 0: # first bin
                    remove_edge = 1 # merge right
                elif idx == len(counts) - 1: # last bin
                    remove_edge = idx # merge left
                else: # choose neighbor with lesser count
                    if counts[idx - 1] <= counts[idx + 1]:
                        remove_edge = idx # merge left
                    else: remove_edge = idx + 1 # merge right

                # Safety check: do not go below 2 bins
                if len(bin_edges) - 1 <= 2: break
                bin_edges = np.delete(bin_edges, remove_edge)

        # Final frequency
        freq = get_counts(x_valid, bin_edges)

        # Include NaN count
        if include_nan:
            freq = np.append(freq, x.size - x_valid.size)
            
        return freq, bin_edges
    
    @staticmethod
    def ordinal_mapping(encoder, columns, unknown_value=-1, missing_value=-2):
    
        '''
        Create category-to-encoded-value mappings from a fitted sklearn 
        OrdinalEncoder, including:

        - regular categories
        - missing values
        - infrequent grouped categories
        - unknown categories

        Parameters
        ----------
        encoder : sklearn.preprocessing.OrdinalEncoder
            Fitted OrdinalEncoder instance.

        columns : list-like 
            Column names corresponding to encoder.categories_.
            If None, generic column names are generated.

        unknown_value : int, default=-1
            Encoded value used for unknown categories.

        Returns
        -------
        mapping : dict
            Nested dictionary mapping:
            {column_name: {category: encoded_value}}

        '''

        # Validate fitted encoder
        if not hasattr(encoder, "categories_"):
            raise ValueError("`encoder` must be fitted first.")

        mapping = dict()

        for n, (col, cats) in enumerate(zip(columns, encoder.categories_)):

            col_map, index = dict(), 0

            # Get infrequent categories for current feature
            infrequent = None
            if hasattr(encoder, "infrequent_categories_"):
                infrequent = encoder.infrequent_categories_[n]

            if infrequent is None: 
                infrequent = []

            # Regular categories
            for cat in cats:

                if pd.isna(cat):
                    # Missing values
                    col_map[np.nan] = missing_value
                elif cat not in infrequent:
                    # Infrequent categories
                    col_map[cat] = index
                    index += 1

            # Add grouped infrequent category
            if infrequent is not None:

                for cat in infrequent:
                    col_map[cat] = index
                col_map["INFREQUENT_VALUE"] = index

                # Unknown category
                col_map["UNKNOWN_VALUE"] = unknown_value

            mapping[col] = col_map

        return mapping

In [6]:
class CalculateStats:
    
    @staticmethod
    def calculate_chi2(f_exp, f_obs, max_stat=999):
        
        '''
        Compute the Chi-Square goodness-of-fit statistic between expected 
        and observed distributions.

        Parameters
        ----------
        f_exp : array-like
            Expected frequencies or proportions.

        f_obs : array-like
            Observed frequencies or proportions.

        max_stat : float, default=999
            Maximum allowed Chi-Square statistic value. This helps prevent 
            numerical overflow and extremely small p-values.

        Returns
        -------
        chi2 : namedtuple
            Named tuple containing:

            - stat : float
                Chi-Square test statistic.

            - df : int
                Degrees of freedom used in the test.

            - pvalue : float
                Upper-tail p-value from the Chi-Square distribution.
                
        Notes
        -----
        Typical interpretation of Chi-Square p-value (CI=95%):
        - p-value >= 0.05 : similar
        - p-value < 0.05  : significant difference
        - p-value < 0.01  : distribution drift
        
        '''
        # Validate input
        f_exp = np.where(f_exp == 0, 1 / sum(f_exp), f_exp)
        f_exp = np.asarray(f_exp).ravel() / sum(f_exp)
        f_obs = np.asarray(f_obs).ravel() / sum(f_obs)
        
        # Chi-Square test for goodness of fit.
        chi2_stat = ((f_obs - f_exp)**2 / f_exp).sum() 
        chi2_stat = float(min(chi2_stat * 100, max_stat)) 
        chi2_df = max(len(f_exp) - 2, 1)
        chi2_pvalue = float(1 - stats.chi2.cdf(chi2_stat, df=chi2_df))
        chi2 = namedtuple("Chi2", ["stat", "df", "pvalue"])
  
        return chi2(chi2_stat, chi2_df, chi2_pvalue)
    
    @staticmethod
    def calculate_psi(f_exp, f_obs):
        
        '''
        Compute the Population Stability Index (PSI) between expected and
        observed distributions.

        Parameters
        ----------
        f_exp : array-like
            Expected frequencies or proportions.

        f_obs : array-like
            Observed frequencies or proportions.

        Returns
        -------
        psi : float
            Population Stability Index.

        Notes
        -----
        Typical PSI interpretation:
        - PSI < 0.10 : negligible drift
        - PSI < 0.25 : moderate drift
        - PSI ≥ 0.25 : significant drift
     
        '''
        # Validate input
        f_exp = np.where(f_exp == 0, 1 / sum(f_exp), f_exp)
        f_exp = np.asarray(f_exp).ravel() / sum(f_exp)
        f_obs = np.asarray(f_obs).ravel() / sum(f_obs)
        f_obs = np.clip(f_obs, np.finfo("float").eps, None)
        
        # Population Stability Index
        psi = (f_obs - f_exp) * np.log(f_obs / f_exp)
        psi = float(np.nan_to_num(psi, nan=0.).sum())
    
        return psi
    
    @staticmethod
    def calculate_ks(f_exp, f_obs):
        
        '''
        Compute the approximate two-sample Kolmogorov-Smirnov (KS)
        statistic using empirical cumulative distributions derived from
        frequency counts.

        Parameters
        ----------
        f_exp : array-like
            Expected frequencies or histogram counts.

        f_obs : array-like
            Observed frequencies or histogram counts.
        
        Returns
        -------
        KS : namedtuple
            Named tuple containing:

            - stat : float
                Kolmogorov-Smirnov statistic representing the maximum
                cumulative distribution difference.

            - pvalue : float
                Approximate p-value of the KS statistic.
            
        Notes
        -----
        Typical interpretation of KS statistic:
        - KS < 0.05 : negligible distribution drift
        - KS < 0.10 : minor drift
        - KS < 0.20 : moderate drift
        - KS ≥ 0.20 : substantial drift

        '''
        # Empirical CDFs
        n1 = sum(f_exp)
        n2 = sum(f_obs)
        cdf1 = np.cumsum(f_exp) / n1
        cdf2 = np.cumsum(f_obs) / n2

        # KS statistic
        ks_stat = float(np.max(np.abs(cdf1 - cdf2)))

        # Effective sample size
        n_eff = np.sqrt((n1 * n2) / (n1 + n2))

        # Asymptotic p-value approximation
        p_value = float(kstwobign.sf(n_eff * ks_stat))
        KS = namedtuple("KS", ["stat", "pvalue"])

        return KS(ks_stat, p_value)
    
    @staticmethod
    def calculate_wsd(f_exp, f_obs, bin_edges):
        
        '''
        Compute the histogram-based Wasserstein distance between expected
        and observed distributions.

        Parameters
        ----------
        f_exp : array-like
            Expected frequencies or histogram counts.

        f_obs : array-like
            Observed frequencies or histogram counts.

        bin_edges : array-like
            Monotonically increasing histogram bin edges with length
            equal to n_bins + 1

        Returns
        -------
        distance : float
            Wasserstein distance between the two distributions.
        
        Notes
        -----
        Typical Interpretation (base=2)
        - wsd ≈ 0    : identical
        - wsd < 0.05 : very stable
        - wsd < 0.10 : mild drift
        - wsd < 0.25 : moderate drift
        - wsd > 0.25 : substantial drift

        '''
        if bin_edges is None:
            return None
            
        # Normalize to probabilities
        p1 = f_exp / f_exp.sum()
        p2 = f_obs / f_obs.sum()

        # A sample from a probability distribution
        centers = (bin_edges[:-1] + bin_edges[1:]) / 2
        
        # Wasserstein distance
        args = (centers, centers, p1, p2)
        distance = float(wasserstein_distance(*args))

        return distance
    
    @staticmethod
    def calculate_jsd(f_exp, f_obs, base=2):
        
        '''
        Compute the Jensen-Shannon divergence (JSD) between expected and
        observed distributions.

        Parameters
        ----------
        f_exp : array-like
            Expected frequencies or histogram counts.

        f_obs : array-like
            Observed frequencies or histogram counts.

        base : float, default=2
            Logarithmic base used in divergence calculation.
            When `base=2`, the output is bounded between 0 and 1.
            
        Returns
        -------
        jsd : float
            Jensen-Shannon divergence between the two distributions.

        Notes
        -----
        Typical Interpretation (base=2)
        - JSD ≈ 0.00 : nearly identical distributions
        - JSD < 0.10 : minor drift
        - JSD < 0.20 : moderate drift
        - JSD ≥ 0.20 : substantial drift
            
        '''
        # Normalize to probabilities
        p1 = f_exp / f_exp.sum()
        p2 = f_obs / f_obs.sum()
        
        # Numerical stability
        eps = np.finfo(np.float64).eps
        p1 = np.clip(p1, eps, None)
        p2 = np.clip(p2, eps, None)

        # Renormalize
        p1 /= p1.sum()
        p2 /= p2.sum()

        jsd = float(jensenshannon(p1, p2, base=base))

        return jsd

In [7]:
@dataclass
class FeatureDistribution:
    freq : np.ndarray
    bin_edges : np.ndarray = None
    categories : dict = None
    dtype : object = None
    include_nan : bool = True
        
@dataclass
class CompareDistribution:
    f_exp : np.ndarray
    f_obs : np.ndarray
    bin_edges : np.ndarray = None
    categories : dict = None
    dtype : object = None
    include_nan : bool = True
    chi2 : object = None
    psi : object = None
    ks : object = None
    wsd : object = None
    jsd : object = None        

In [8]:
class CompareDistributions(BaseEstimator, TransformerMixin, 
                           ValidateParams, Binning, CalculateStats):
    
    '''
    Compare feature distributions between a baseline dataset and a
    new observation dataset.

    This class supports both numerical and categorical variables and
    computes statistical drift metrics between an expected distribution
    (`fit`) and an observed distribution (`compare`).

    Numerical features are binned before comparison, while categorical
    features are ordinally encoded with optional grouping of infrequent
    categories.

    Parameters
    ----------
    n_bins : int, default=10
        Number of bins used for automatic numerical binning.

     equal_width : bool, default=True
            If True, uses equal-width binning. Otherwise, uses quantile
            based binning.

    normal_dist : bool, default=True
        If True, applies a quantile transformation to approximate a 
        normal distribution before binning.
            
    include_nan : bool, default=True
        Whether missing values should be included as a separate bin.

    n_min : int or float, default=30
        Minimum required observations per bin.
        - If int >= 1: Minimum absolute count per bin.
        - If 0 < float < 1: Minimum fraction of total non-NaN samples.
        - Categories below the threshold are merged into an infrequent
          category.

    max_categories : int, default=None
        Maximum number of categorical output groups.If infrequent 
        categories exist, they are counted as one additional category. 
        If None, all categories are retained.

    Attributes
    ----------
    baseline_numeric_ : dict
        Baseline distributions for numerical variables.

    baseline_categorical_ : dict
        Baseline distributions for categorical variables.

    compare_numeric_ : dict
        Comparison results for numerical variables.

    compare_categorical_ : dict
        Comparison results for categorical variables.

    mapping_ : dict
        Ordinal encoding mapping for categorical variables.

    '''
    
    def __init__(self, n_bins=10, equal_width=True, normal_dist=False, 
                 include_nan=True, n_min=30, max_categories=None):
        
        # Numerical binning configuration
        self.n_bins = self.Interval("n_bins", n_bins, int, 2, None, "left")
        
        # Boolean parameters
        args = ([True, False], bool)
        self.equal_width = self.StrOptions("equal_width", equal_width, *args)
        self.normal_dist = self.StrOptions("normal_dist", normal_dist, *args)
        self.include_nan = self.StrOptions("include_nan", include_nan, *args)
        
        
        # Minimum categorical frequency threshold
        if isinstance(n_min, int):
            self.n_min = self.Interval("n_min", n_min, int, 0, None, "left")
        elif isinstance(n_min, float):
            self.n_min = self.Interval("n_min", n_min, float, 0, 1, "neither")
        else: raise ValueError("`n_min` must be either int or float.")
        
        # Maximum categorical output groups
        if max_categories is not None:
            args = ("max_categories", max_categories, int, 2, None, "left")
            self.max_categories = self.Interval(*args)
        else: self.max_categories = max_categories
        
        # Encoder for categorical variables
        self.encoder = OrdinalEncoder(categories='auto', 
                                      dtype=np.int16, 
                                      unknown_value=-1,
                                      handle_unknown="use_encoded_value", 
                                      encoded_missing_value=-2, 
                                      min_frequency=self.n_min, 
                                      max_categories=self.max_categories)
    
    def fit(self, X, bins=None):
        
        '''
        Fit baseline feature distributions from a reference dataset.

        Parameters
        ----------
        X : pandas.DataFrame
            Input dataframe used as the baseline or reference dataset.

        bins : dict, default=None
            Dictionary of custom bins for numerical variables. If not 
            provided for a specific numeric column, bins are determined
            automactially. Example: {"age": [0, 18, 35, 60, 100]}, where 
            key is column name and value is a list of scalars defining 
            the bin edges.
            
        Returns
        -------
        self : CompareDistributions
            Fitted estimator containing baseline distributions and
            feature metadata.
        
        '''
        if not isinstance(X, pd.DataFrame):
            raise TypeError(f"`X` must be pd.DataFrame. Got {type(X)}.")
            
        # Store custom bins
        self.bins = {} if bins is None else bins
            
        # Numeric and catigorical features from expected observation
        self.num_features_ = X.select_dtypes(include=np.number).columns
        self.cat_features_ = X.select_dtypes(include="category").columns
        
        # Store feature dtypes
        self.dtypes_ = X.dtypes.to_dict()
        
        # Initialize baseline containers
        self.baseline_numeric_ = OrderedDict()
        self.baseline_categorical_ = OrderedDict()
    
        # Fit numerical variables
        if len(self.num_features_): 
            self._fit_numeric(X)
        
        # Fit categorical variables
        if len(self.cat_features_): 
            self._fit_categorical(X)
        
        return self
    
    def _fit_numeric(self, X):
        
        '''
        Fit baseline distributions for numerical variables.

        Parameters
        ----------
        X : pandas.DataFrame
            Input dataframe containing numerical features.

        Attributes
        ----------
        self.baseline_numeric_ : dict
            Baseline numerical distributions.
    
        '''        
        for var in self.num_features_:
            
            # Calculate bin edges, given binning method.
            bin_edges = self.bins.get(var, None)
            if bin_edges is None:

                # Generate bin edges
                kwargs = dict(n_bins=self.n_bins, 
                              equal_width=self.equal_width, 
                              normal_dist=self.normal_dist)
                bin_edges = self.create_numeric_bins(X[var], **kwargs)
            
            else: bin_edges = self.check_bins(bin_edges)
            
            # Compute frequency
            kwargs = dict(include_nan=True, n_min=self.n_min, merge=True)
            freq, bin_edges = self.compute_frequency(X[var], bin_edges, **kwargs)
            
            # Baseline numerical distributions
            args = (freq, bin_edges, None, self.dtypes_[var], self.include_nan)
            self.baseline_numeric_[var] = FeatureDistribution(*args)
        
    def _fit_categorical(self, X):
        
        '''
        Fit baseline distributions for categorical variables. Infrequent 
        categories may optionally be grouped together based on `n_min` 
        and `max_categories`.

        Parameters
        ----------
        X : pandas.DataFrame
            Input dataframe containing numerical features.

        Attributes
        ----------
        self.baseline_categorical_ : dict
            Baseline categorical distributions.
        
        '''  
        # Calculate bin edges,
        self.encoder.fit(X[self.cat_features_])
        self.mapping_ = self.ordinal_mapping(self.encoder, 
                                             self.cat_features_, 
                                             unknown_value=-1, 
                                             missing_value=-2)

        # Categorical masking
        X_cat = self.encoder.transform(X[self.cat_features_])

        for n, var in enumerate(self.cat_features_):

            # Summarize categories
            key = self.cat_features_[n]
            categories = dict()
            for k,v in self.mapping_[key].items():
                if v in categories.keys():
                    if not isinstance(categories[v], list):
                        categories[v] = [categories[v]]
                    categories[v] += [k]
                else: categories[v] = k

            # Compute frequency
            x = X_cat[:,n]
            freq = np.array([int(sum(x==code)) for code in 
                             range(-2, max(categories.keys()) + 1)])

            # Baseline categorical distributions
            args = (freq, None, dict(sorted(categories.items())), "category")
            self.baseline_categorical_[var] = FeatureDistribution(*args)
            
    def compare(self, X):
        
        '''
        Compare feature distributions against the fitted baseline.

        Parameters
        ----------
        X : pandas.DataFrame
            Input dataframe containing features to compare against the
            fitted baseline distributions.

        Returns
        -------
        self : CompareDistributions
            Estimator containing comparison results and statistical
            drift metrics.

        Attributes
        ----------
        Comparison results are stored in:
            - `self.compare_numeric_`
            - `self.compare_categorical_`

        '''
        # Validation
        if not hasattr(self, "baseline_numeric_"):
            raise AttributeError("This transformer is not fitted yet. "
                                 "Call `fit()` before `compare()`.")

        if not isinstance(X, pd.DataFrame):
            raise TypeError(f"`X` must be pd.DataFrame. Got {type(X)}.")
        
        # Numeric and catigorical features from actual observation
        num_features = list(set(self.num_features_).intersection(list(X)))
        cat_features = list(set(self.cat_features_).intersection(list(X)))
        
        # Baseline results
        self.compare_numeric_ = OrderedDict()
        self.compare_categorical_ = OrderedDict()
        
        # Compare numerical variables
        if len(num_features):
            self._compare_numeric(X[num_features])
        
        # Compare categorical variables
        if len(cat_features):
            self._compare_categorical(X[cat_features])
        
        return self

    def _compare_numeric(self, X):
        
        '''
        Compare numerical feature distributions against the baseline.

        Parameters
        ----------
        X : pandas.DataFrame
            Input dataframe containing numerical features to compare
            against the fitted baseline.

        Attributes
        ----------
        self.compare_numeric_ : dict
            Comparison results.

        '''
        for var in X.columns:
            
            # Get bin edges
            baseline = self.baseline_numeric_[var]
            
            f_exp = baseline.freq
            if self.include_nan==False: 
                f_exp = f_exp[:-1]
            else: 
                bin_edges = baseline.bin_edges
                last_edge = bin_edges[-1] + np.mean(np.diff(bin_edges))
                bin_edges = np.append(bin_edges, last_edge)
            
            # Compute frequency
            kwargs = dict(include_nan=self.include_nan, merge=False)
            f_obs = self.compute_frequency(X[var], baseline.bin_edges, **kwargs)[0]
            
            # Metrics
            metrics = self._calculate_metrics(f_exp, f_obs, bin_edges)
            
            # Baseline numerical distributions
            args = (f_exp, f_obs, bin_edges, None, baseline.dtype, self.include_nan)
            self.compare_numeric_[var] = CompareDistribution(*args, **metrics)
    
    def _compare_categorical(self, X):
        
        '''
        Compare categorical feature distributions against the baseline.

        Parameters
        ----------
        X : pandas.DataFrame
            Input dataframe containing categorical features to compare
            against the fitted baseline.

        Attributes
        ----------
        self.compare_categorical_ : dict
            Comparison results.

        '''
        for var in X.columns:

            # Get bin edges
            baseline = self.baseline_categorical_[var]
            f_exp = baseline.freq

            func = self.mapping_[var]
            x = X[var].map(func).fillna(-1)
            f_obs = np.array([int(sum(x==code)) for code in 
                              range(-2, max(func.values())+1)])

            # Metrics
            metrics = self._calculate_metrics(f_exp, f_obs, None)

            # Baseline numerical distributions
            args = (f_exp, f_obs, None, baseline.categories, 
                    baseline.dtype, self.include_nan)
            self.compare_categorical_[var] = CompareDistribution(*args, **metrics)
                
    def _calculate_metrics(self, f_exp, f_obs, bin_edges=None):
        
        '''
        Compute statistical distance and drift metrics between expected 
        and observed distributions.

        Parameters
        ----------
        f_exp : array-like
            Expected frequencies or histogram counts.

        f_obs : array-like
            Observed frequencies or histogram counts.

        bin_edges : array-like, default=None
            Monotonically increasing histogram bin edges with length
            equal to n_bins + 1. Ignored for categorical variables.

        Returns
        -------
        dict
            Dictionary containing computed statistical metrics.
            
            Returned keys include:
            - chi2 : Chi-Square statistic
            - psi  : Population Stability Index
            - ks   : Kolmogorov-Smirnov statistic
            - wsd  : Wasserstein distance
            - jsd  : Jensen-Shannon distance
            
        '''
        return {"chi2": self.calculate_chi2(f_exp, f_obs),
                "psi" : self.calculate_psi(f_exp, f_obs),
                "ks"  : self.calculate_ks(f_exp, f_obs),
                "wsd" : self.calculate_wsd(f_exp, f_obs, bin_edges),
                "jsd" : self.calculate_jsd(f_exp, f_obs)}

In [ ]:
def __info__(self):
        
        '''self.info : pd.DataFrame'''
        data = []
        attr = lambda k,f : getattr(self.result[k],f)
        for key in self.result.keys():
            stats = [attr(key,fld) if fld!='chi2_bins' 
                     else 0 for fld in self.fields]
            data.append([key] + stats)
        info = pd.DataFrame(data, columns=["variable"] + self.fields)
        info['chi2_bins'] = info['chi2_df'] + 1
        self.info = info.set_index("variable")

### Example

In [9]:
from sklearn.ensemble import RandomForestClassifier as rf
from sklearn.model_selection import train_test_split as tts
from sklearn.datasets import make_classification, fetch_openml

In [10]:
adult = fetch_openml(data_id=1590, as_frame=True, parser='auto').frame.drop(columns=["fnlwgt","education-num"])

In [11]:
for col in adult.columns:
    nan = np.random.random(size=len(adult))<=0.1
    adult.loc[nan, col] = np.nan

In [12]:
model1 = OptimizeDtypes()
model1.fit(adult)
adult_new = model1.transform(adult)

In [13]:
model1.memory_report

{'before_mb': 1.9195,
 'after_mb': 0.9879,
 'delta_mb': 0.9316,
 'delta_pct': 48.53}

In [14]:
model1.dtypes_

{'age': Int8Dtype(),
 'workclass': 'category',
 'education': 'category',
 'marital-status': 'category',
 'occupation': 'category',
 'relationship': 'category',
 'race': 'category',
 'sex': 'category',
 'capital-gain': Int32Dtype(),
 'capital-loss': Int16Dtype(),
 'hours-per-week': Int8Dtype(),
 'native-country': 'category',
 'class': 'category'}

In [15]:
a = CompareDistributions()

In [16]:
a.fit(adult)

CompareDistributions()

In [17]:
a.baseline_numeric_['capital-gain']

FeatureDistribution(freq=array([42905,   676,   119,   222,  4920]), bin_edges=array([    0. ,  9999.9, 19999.8, 89999.1, 99999. ]), categories=None, dtype=dtype('float64'), include_nan=True)

In [18]:
a.baseline_categorical_['native-country']

FeatureDistribution(freq=array([ 5643,     0,   163,   111,    75,   123,    93,    42,   138,
         118,    33,   184,    44,    76,    66,   139,    53,    35,
         100,    95,    82,   865,    45,    43,   267,    76,    57,
         166,   106,    62, 39446,    73,   223]), bin_edges=None, categories={-2: nan, -1: 'UNKNOWN_VALUE', 0: 'Canada', 1: 'China', 2: 'Columbia', 3: 'Cuba', 4: 'Dominican-Republic', 5: 'Ecuador', 6: 'El-Salvador', 7: 'England', 8: 'France', 9: 'Germany', 10: 'Greece', 11: 'Guatemala', 12: 'Haiti', 13: 'India', 14: 'Iran', 15: 'Ireland', 16: 'Italy', 17: 'Jamaica', 18: 'Japan', 19: 'Mexico', 20: 'Nicaragua', 21: 'Peru', 22: 'Philippines', 23: 'Poland', 24: 'Portugal', 25: 'Puerto-Rico', 26: 'South', 27: 'Taiwan', 28: 'United-States', 29: 'Vietnam', 30: ['Cambodia', 'Holand-Netherlands', 'Honduras', 'Hong', 'Hungary', 'Laos', 'Outlying-US(Guam-USVI-etc)', 'Scotland', 'Thailand', 'Trinadad&Tobago', 'Yugoslavia', 'INFREQUENT_VALUE']}, dtype='category', inc

In [19]:
a.compare(adult.sample(10000).reset_index(drop=True))

CompareDistributions()

In [20]:
a.compare_categorical_['class']

CompareDistribution(f_exp=array([ 4832,     0, 33492, 10518,     0]), f_obs=array([ 991,    0, 6860, 2149,    0]), bin_edges=None, categories={-2: nan, -1: 'UNKNOWN_VALUE', 0: '<=50K', 1: '>50K', 2: 'INFREQUENT_VALUE'}, dtype='category', include_nan=True, chi2=Chi2(stat=0.00013316575746245677, df=3, pvalue=0.9999995913134901), psi=1.3436336098849199e-06, ks=KS(stat=0.00044744686949760304, pvalue=1.0), wsd=None, jsd=0.0004900222379765287)

In [21]:
@dataclass
class PlotParams:
    include_nan : bool = True
    colors : list = None 
    labels : list = None
    tight_layout : bool = True 
    decimal : int = 0 
    expect_kwds : dict = None 
    observe_kwds : dict = None
    xticklabel_format = None 
    max_display : int = 2

In [ ]:
class PlotDistributions(BaseEstimator, TransformerMixin, 
                        ValidateParams):
    
    bar_kwargs  = dict(width=0.4, alpha=0.9)
    anno_kwargs = dict(xytext=(0,4), 
                       textcoords='offset points', 
                       va='bottom', 
                       ha='center', 
                       fontsize=13, 
                       fontweight='demibold')
    
    colors = ["#1B9CFC", "#FC427B"]
    
    
    
    
    def plot(self, compare, var, ax=None, labels=None,
             tight_layout=True, decimal=0, 
             expect_kwds=None, 
             observe_kwds=None, 
             xticklabel_format=None, 
             max_display=2):
        
        if expect_kwds is None: expect_kwds = dict()
        if observe_kwds is None: observe_kwds = dict()
            
        self.num_format = ("{:,." + str(decimal) + "%}").format
        
        is_numeric = var in self.compare_numeric_.keys()
        is_categorical = var in self.compare_categorical_.keys()
        
        if (is_numeric==False) & (is_categorical==False):
            raise ValueError(f"`var` ('{var}') must be in `compare_numeric_`"
                             f" or `compare_categorical_`.")
        elif is_numeric:
            compare = getattr(self, "compare_numeric_")[var]
        else: compare = getattr(self, "compare_categorical_")[var]
        
        f_exp = compare.f_exp
        f_obs = compare.f_obs
        dtype = compare.dtype

        x = np.arange(len(f_exp))
    
        # Create matplotlib.axes if ax is None.
        width = max(6.5, len(x) * 0.9)
        ax = plt.subplots(figsize=(width, 4.5))[1] if ax is None else ax

    
    def plot_vbar(self, ax, compare):
        
        f_exp = compare.f_exp
        f_obs = compare.f_obs
   
        
        # Bar plots

        anno_kwds = dict((n, {**self.annotation_kwargs, **dict(color=c)}) 
                         for n,c in enumerate(self.colors))
        


        # Vertical bar (Expect).
        {color=colors[0], label ='{} ({:,d})'.format("Expect", int(sum(f_exp)))}
        bar_kwargs
        
        
        kwds = dict()
        
        ax.bar(x-0.25, f_exp, **({**kwds, **expect_kwds} if expect_kwds is not None else kwds))
      
        # Annotation (Expect).
        kwds = {**anno_kwds, **dict(color=colors[0])}
        for xy in zip(x - 0.25, f_exp): 
            ax.annotate(num_format(min(xy[1],1)), xy, **kwds)

        # Vertical bar (Observe).   
        kwds = dict(width=0.4, alpha=0.9, color=colors[1], 
                    label='{} ({:,d})'.format(labels[1], compare.n_[1]))    
        ax.bar(x+0.25, f_obs, **({**kwds, **observe_kwds} if 
                                 observe_kwds is not None else kwds))

        # Annotation (Observe).
        kwds = {**anno_kwds, **dict(color=colors[1])}
        for xy in zip(x + 0.25, f_obs): 
            ax.annotate(num_format(min(xy[1],1)), xy, **kwds)
    
    def __format__(self, ax):
        
        '''
        '''
        # color the y tick labels that have the feature values as gray
        # (these fall behind the black ones with just the feature name)
        for label in aax.xaxis.get_majorticklabels():
            label.set_color("#999999")

        for spine in ["top", "left", "right"]:
            ax.spines[spine].set_visible(False)
            
        ax.yaxis.set_visible(False)
        ax.patch.set_alpha(0)
        
        y_min, y_max = ax.get_ylim()
        ax.set_ylim(y_min, y_max / 0.7)
        ax.set_xlim(-0.5, len(x) - 0.5)

        ax.legend(edgecolor="grey", 
                  facecolor="none", 
                  borderaxespad=0.5, 
                  markerscale=1, 
                  columnspacing=0.3, 
                  handletextpad=0.5, 
                  loc="upper right", 
                  prop=dict(size=12)) 

In [ ]:
def plotting_2samp_base(compare, var, ax=None, colors=None, labels=None,
                        tight_layout=True, decimal=0, expect_kwds=None, 
                        observe_kwds=None, xticklabel_format=None, 
                        max_display=2):
        
    '''
    Plot Chi-Square Goodness of Fit Test.

    Parameters
    ----------
    compare : class object
        Fitted `Compare2samp` object.
    
    var : str
        Variable name in self.info (attribute).

    ax : Matplotlib axis object, default=None
        Predefined Matplotlib axis. If None, `ax` is created with 
        default figsize.

    colors : list of color-hex, default=None
        Number of color-hex must be greater than 1. If None, it uses 
        default colors from Matplotlib.
        
    labels : list of str, default=None
        A sequence of strings providing the labels for each class i.e. 
        "0", and "1". If None, it defaults to ["Expect", "Observe"].
        
    tight_layout : bool, default=True
        If True, it adjusts the padding between and around subplots 
        i.e. plt.tight_layout().

    decimal : int, default=0
        Decimal places for annotation of value(s).

    expect_kwds : keywords, default=None
        Keyword arguments of expected samples to be passed to "ax.bar".

    observe_kwds : keywords, default=None
        Keyword arguments of observed samples to be passed to "ax.bar".

    xticklabel_format : string formatter, default=None
        String formatters (function) for ax.xticklabels values. If None, 
        it defaults to "{:,.xf}".format, where x is a decimal place
        determined automatically by algorithm. If x exceeds 5, it
        defaults to "{:,.1e}".format.

    max_display : int, default=1
        Maximum number of categories to be displayed. This is available 
        only when dtype=="category".

    Returns
    -------
    ax : Matplotlib axis object

    '''
    # ===============================================================
    # Get values from self.hist_data
    var_list = list(compare.hist_data.keys())
    if var not in var_list:
        raise ValueError(f"var must be in {var_list}. "
                         f"Got {var} instead.")
    f_exp = compare.hist_data[var].f_exp
    f_obs = compare.hist_data[var].f_obs
    dtype = compare.hist_data[var].type
    data  = compare.result[var]
    bins  = data.chi2_bins
    x = np.arange(len(f_exp))
    # ---------------------------------------------------------------
    # Create matplotlib.axes if ax is None.
    width  = max(6.5, len(f_exp) * 0.9)
    if ax is None: ax = plt.subplots(figsize=(width, 4.5))[1] 
    colors = ([ax._get_lines.get_next_color() for n in range(2)] 
              if colors is None else colors)
    # ===============================================================

    # Bar plots
    # ===============================================================
    num_format = ("{:,." + str(decimal) + "%}").format
    anno_kwds = dict(xytext =(0,4), textcoords='offset points', 
                     va='bottom', ha='center', fontsize=13, 
                     fontweight='demibold')
    if labels is None: labels = ["Expect", "Observe"]
    # ---------------------------------------------------------------
    # Vertical bar (Expect).
    kwds = dict(width=0.4, alpha=0.9, color=colors[0], 
                label='{} ({:,d})'.format(labels[0], compare.n_[0]))
    ax.bar(x-0.25, f_exp, **({**kwds, **expect_kwds} if 
                             expect_kwds is not None else kwds))
    # ---------------------------------------------------------------
    # Annotation (Expect).
    kwds = {**anno_kwds, **dict(color=colors[0])}
    for xy in zip(x-0.25, f_exp): 
        ax.annotate(num_format(min(xy[1],1)), xy, **kwds)
    # ---------------------------------------------------------------
    # Vertical bar (Observe).   
    kwds = dict(width=0.4, alpha=0.9, color=colors[1], 
                label='{} ({:,d})'.format(labels[1], compare.n_[1]))    
    ax.bar(x+0.25, f_obs, **({**kwds, **observe_kwds} if 
                             observe_kwds is not None else kwds))
    # ---------------------------------------------------------------
    # Annotation (Observe).
    kwds = {**anno_kwds, **dict(color=colors[1])}
    for xy in zip(x + 0.25, f_obs): 
        ax.annotate(num_format(min(xy[1],1)), xy, **kwds)
    # ===============================================================

    # Set other attributes         
    # ===============================================================
    title  = f"Variable : {var}\n" 
    pvalue = lambda v : "N/A" if np.isnan(v) else "{:,.0%}".format(v)
    args   = (pvalue(data.chi2_pvalue), pvalue(data.ks_pvalue)) 
    title += r"p-value ($\chi^{2}$, KS) : " + "({}, {})".format(*args)
    title += "\nPSI : {:.4f}".format(data.psi)
    props = dict(boxstyle='square', facecolor='none', alpha=0)
    ax.text(0, 0.97, title, transform=ax.transAxes, fontsize=13,
            va='top', ha="left", bbox=props)
    # ---------------------------------------------------------------
    # x-ticklabels number format
    if (xticklabel_format is None) & (dtype=="number"): 
        pts = 0
        while True:
            adj = np.unique(np.round(bins, pts))
            if len(adj)<len(bins): pts += 1
            else: break
        if pts > 5: n_format = "{:,.1e}".format
        n_format = ("{:,." + str(pts) + "f}").format
    else: n_format = xticklabel_format
    # ---------------------------------------------------------------   
    xticklabels = []
    if dtype=="number":
        for n in np.arange(len(f_exp)):
            if n < len(f_exp)-1: 
                r = r"$<$"+ f"{n_format(bins[n+1])}"
            else: r = (r"$\geq$" + f"{n_format(bins[-1])}" 
                       + "\nor missing")
            xticklabels.append(f"Group {n+1}" + "\n" + r)
    # ---------------------------------------------------------------
    elif dtype=="category":
        for n, m in bins:
            if max_display>0:
                # format = {A, B,...(n)}
                n_set = [f'"{s}"' for s in np.array(m)[:max_display]]
                if len(m)>max_display: n_set += [" ..({:,d})".format(len(m))]
                xticklabels += [f"Group {n}" + "\n{" + ",".join(n_set) + "}"]
            else: xticklabels += [f"Group {n}" + "\n({:,.0f})".format(len(m))]
    # ---------------------------------------------------------------            
    xticklabels += [label.split('\n')[0] for label in xticklabels]
    xtick_pos = (list(x) + list(x+1e-8))
    plt.xticks(xtick_pos, xticklabels, fontsize=12)
    # ---------------------------------------------------------------
    # color the y tick labels that have the feature values as gray
    # (these fall behind the black ones with just the feature name)
    tick_labels = ax.xaxis.get_majorticklabels()
    for i in x: tick_labels[i].set_color("#999999")
    ax.set_xlim(-0.5, len(f_exp)-0.5)
    # ---------------------------------------------------------------
    for spine in ["top", "left", "right"]:
        ax.spines[spine].set_visible(False)
    ax.yaxis.set_visible(False)
    ax.patch.set_alpha(0)
    y_min, y_max = ax.get_ylim()
    ax.set_ylim(y_min, y_max/0.7)
    # ---------------------------------------------------------------
    ax.legend(edgecolor="grey", facecolor="none", borderaxespad=0.5, 
              markerscale=1, columnspacing=0.3,handletextpad=0.5, 
              loc="upper right", prop=dict(size=12)) 
    if tight_layout: plt.tight_layout()
    # ===============================================================
    
    return ax